### Imports

In [1]:
import json
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sn

print(pg.__version__) # 0.5.3
print(pd.__version__) # 2.0.3
print(np.__version__) # 1.24.3
print(sn.__version__) # 0.13.0

from utils_MS import *

# %load_ext autotime

0.5.3
2.2.2
1.26.4
0.13.2


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
# def run(exp):

### Parameters

In [3]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"]

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

has_transformation = params["has_transformation"]
print("Has transformation:", has_transformation)

threshold_corr = params["threshold_corr"]
print("Threshold corr:\t", threshold_corr)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups_id_no = params["groups_id_no"]
print("Groups id (no):\t", groups_id_no)

Exp:		 exp11
Data variations: ['none']
Has transformation: True
Threshold corr:	 0.5
Groups id:	 ['AR', 'CRS', 'OSA', 'LPRD', 'SGB', 'LSNB', 'RCC', 'BC', 'BPH', 'PCa', 'PD']
Subgroups id:	 {'AR': ['1', '2'], 'CRS': ['1', '2'], 'OSA': ['1', '2'], 'LPRD': ['1', '2'], 'SGB': ['1', '2'], 'LSNB': ['1', '2'], 'RCC': ['1', '2'], 'BC': ['1', '2'], 'BPH': ['1', '2'], 'PCa': ['1', '2'], 'PD': ['1', '2']}
Groups id (no):	 ['Blank', 'QC', 'Std']


In [4]:
# Remove
# groups_id = ["OSA"]

### Load dataset

In [5]:
# read raw data
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0)
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
0,1,69.99951,Unknown,3.761231,0.006085,0.013273,3.028029e-02,0.039917,0.050244,0.061227,...,9.724785,0.141547,0.982856,2.836874,2.701034,0.902734,2.906096,1.066105,0.080514,0.084062
1,1,70.04025,Unknown,330.902463,3.345954,7.288531,1.660198e+01,21.875098,27.522634,33.526638,...,1120.030889,929.073344,402.726307,1160.112216,1104.663489,369.947418,1188.392767,436.767183,27220.117777,114.051237
2,1,70.04151,Unknown,3.810483,0.323929,0.609776,1.190282e+00,1.489258,1.794734,2.106832,...,47.784825,87.042228,13.025668,30.770177,29.569910,12.157380,31.378343,13.913295,3.330658,71.881975
3,1,70.04908,Unknown,371.971786,0.779920,1.640930,3.603131e+00,4.689429,5.839962,7.051638,...,245.177154,51.237744,73.230163,201.238003,192.039249,67.524286,205.915956,79.133432,4.764968,4.997353
4,1,70.06267,Unknown,14.383680,0.705716,1.341343,1.985821e+00,2.645148,3.321009,4.013655,...,398.483185,24.863943,23.424482,56.887913,54.609726,21.803690,58.041683,25.084185,13.415288,598.177399
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1,732.79951,Unknown,163.972116,0.257935,0.408037,6.627199e-01,0.779651,17223.840695,0.892606,...,9.504735,7.849464,3.397348,6.601916,6.400000,3.210185,6.703166,3.585089,1.286116,1.508308
5440,1,748.76437,Unknown,5.318633,0.037041,0.088542,2.225000e-01,0.302977,0.391787,0.488619,...,14.683176,12.248418,7.863031,25.008636,23.659197,7.150230,25.699223,8.610731,1.600811,1.899722
5441,1,794.79590,Unknown,44.796079,0.062477,0.158303,4.230776e-01,0.588125,0.773695,0.979264,...,10.908876,9.339637,16.326008,57.273009,53.944851,14.716346,58.983427,18.026856,2.112711,2.430245
5442,1,800.81295,Unknown,32.469829,0.091943,0.246786,7.009871e-01,0.994535,1.330761,1.709149,...,21.954321,19.337485,35.798102,132.791415,124.703890,32.144006,136.955665,39.678934,5.750558,6.534013


In [6]:
# get metadata
df_join_raw_metadata = df_join_raw.iloc[:, :2]
df_join_raw_metadata

,Average Rt,Average Mz
0,1,69.99951
1,1,70.04025
2,1,70.04151
3,1,70.04908
4,1,70.06267
...,...,...
5439,1,732.79951
5440,1,748.76437
5441,1,794.79590
5442,1,800.81295


In [7]:
# filter by samples
columns_sample = [column for column in df_join_raw.columns if column.split("_")[0] not in groups_id_no]
df_join_raw_intensity = df_join_raw.loc[:, columns_sample]
df_join_raw_intensity = df_join_raw_intensity.iloc[:, 3:]
df_join_raw_intensity

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
0,3.761231,0.006085,0.013273,3.028029e-02,0.039917,0.050244,0.061227,0.072840,0.125291,0.170628,...,9.724785,0.141547,0.982856,2.836874,2.701034,0.902734,2.906096,1.066105,0.080514,0.084062
1,330.902463,3.345954,7.288531,1.660198e+01,21.875098,27.522634,33.526638,39.873100,68.517262,93.256691,...,1120.030889,929.073344,402.726307,1160.112216,1104.663489,369.947418,1188.392767,436.767183,27220.117777,114.051237
2,3.810483,0.323929,0.609776,1.190282e+00,1.489258,1.794734,2.106832,2.425549,3.765650,4.837489,...,47.784825,87.042228,13.025668,30.770177,29.569910,12.157380,31.378343,13.913295,3.330658,71.881975
3,371.971786,0.779920,1.640930,3.603131e+00,4.689429,5.839962,7.051638,8.322044,13.959504,18.740449,...,245.177154,51.237744,73.230163,201.238003,192.039249,67.524286,205.915956,79.133432,4.764968,4.997353
4,14.383680,0.705716,1.341343,1.985821e+00,2.645148,3.321009,4.013655,4.723131,7.726094,10.145653,...,398.483185,24.863943,23.424482,56.887913,54.609726,21.803690,58.041683,25.084185,13.415288,598.177399
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,163.972116,0.257935,0.408037,6.627199e-01,0.779651,17223.840695,0.892606,1.002628,1.321296,1.629596,...,9.504735,7.849464,3.397348,6.601916,6.400000,3.210185,6.703166,3.585089,1.286116,1.508308
5440,5.318633,0.037041,0.088542,2.225000e-01,0.302977,0.391787,0.488619,0.593280,1.087502,1.535642,...,14.683176,12.248418,7.863031,25.008636,23.659197,7.150230,25.699223,8.610731,1.600811,1.899722
5441,44.796079,0.062477,0.158303,4.230776e-01,0.588125,0.773695,0.979264,1.204537,2.299270,2.962102,...,10.908876,9.339637,16.326008,57.273009,53.944851,14.716346,58.983427,18.026856,2.112711,2.430245
5442,32.469829,0.091943,0.246786,7.009871e-01,0.994535,1.330761,1.709149,2.129512,4.230971,6.255039,...,21.954321,19.337485,35.798102,132.791415,124.703890,32.144006,136.955665,39.678934,5.750558,6.534013


In [8]:
df_join_raw_intensity.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5444 entries, 0 to 5443
Columns: 308 entries, AR_1.1 to PD_2.14
dtypes: float64(308)
memory usage: 12.8 MB


In [9]:
check_dataset(df_join_raw_intensity)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 1676752
Count greater than 1:	 1614821
Count less than -1:	 0


### Generate graphs

In [10]:
# Transformation (log10)

if not has_transformation:
	df_join_raw_log = log10_global(df_join_raw_intensity)
else:
	df_join_raw_log = df_join_raw_intensity.copy()
df_join_raw_log.head()

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
0,3.761231,0.006085,0.013273,0.030280,0.039917,0.050244,0.061227,0.072840,0.125291,0.170628,...,9.724785,0.141547,0.982856,2.836874,2.701034,0.902734,2.906096,1.066105,0.080514,0.084062
1,330.902463,3.345954,7.288531,16.601984,21.875098,27.522634,33.526638,39.873100,68.517262,93.256691,...,1120.030889,929.073344,402.726307,1160.112216,1104.663489,369.947418,1188.392767,436.767183,27220.117777,114.051237
2,3.810483,0.323929,0.609776,1.190282,1.489258,1.794734,2.106832,2.425549,3.765650,4.837489,...,47.784825,87.042228,13.025668,30.770177,29.569910,12.157380,31.378343,13.913295,3.330658,71.881975
3,371.971786,0.779920,1.640930,3.603131,4.689429,5.839962,7.051638,8.322044,13.959504,18.740449,...,245.177154,51.237744,73.230163,201.238003,192.039249,67.524286,205.915956,79.133432,4.764968,4.997353
4,14.383680,0.705716,1.341343,1.985821,2.645148,3.321009,4.013655,4.723131,7.726094,10.145653,...,398.483185,24.863943,23.424482,56.887913,54.609726,21.803690,58.041683,25.084185,13.415288,598.177399


In [11]:
check_dataset(df_join_raw_log)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 1676752
Count greater than 1:	 1614821
Count less than -1:	 0


In [12]:
# split graph in groups and subgroups

""" def split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id, by_group=False):
	list_df_groups_subgroups = []
	for group in groups_id:
		df_aux = df_join_raw_log.filter(like=group)
		list_aux = []
		
		if by_group:
			list_aux.append(df_aux)
		else:
			for subgroup in subgroups_id[group]:
				list_aux.append(df_aux.filter(like="{}_{}.".format(group, subgroup)))
		list_df_groups_subgroups.append(list_aux)
	return list_df_groups_subgroups """

dict_df_groups_subgroups = split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id)
dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,AR_1.11,AR_1.12,AR_1.13,AR_1.14
0,3.761231,0.006085,0.013273,0.030280,0.039917,0.050244,0.061227,0.072840,0.125291,0.170628,0.186844,0.203610,0.238759,0.257140
1,330.902463,3.345954,7.288531,16.601984,21.875098,27.522634,33.526638,39.873100,68.517262,93.256691,102.103352,111.247433,130.412734,140.433267
2,3.810483,0.323929,0.609776,1.190282,1.489258,1.794734,2.106832,2.425549,3.765650,4.837489,5.207149,5.582902,6.352578,6.746368
3,371.971786,0.779920,1.640930,3.603131,4.689429,5.839962,7.051638,8.322044,13.959504,18.740449,20.435723,22.180943,25.818440,27.710860
4,14.383680,0.705716,1.341343,1.985821,2.645148,3.321009,4.013655,4.723131,7.726094,10.145653,10.983228,11.835865,13.585950,14.483382


In [13]:
check_dataset(dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 76216
Count greater than 1:	 67552
Count less than -1:	 0


In [14]:
# Transpose
dict_groups_subgroups_t = transpose_global(dict_df_groups_subgroups)
dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,3.761231,330.902463,3.810483,371.971786,14.383680,18.712422,417.147841,436.415332,447.095192,546.474624,...,44.344524,97.694474,0.490535,17.259174,74.363530,163.972116,5.318633,44.796079,32.469829,8.757094e+01
1,0.006085,3.345954,0.323929,0.779920,0.705716,1.268849,2.415183,2.922806,2.590657,3.762704,...,1.133287,0.837684,0.009950,0.498310,0.545720,0.257935,0.037041,0.062477,0.091943,1.610831e-02
2,0.013273,7.288531,0.609776,1.640930,1.341343,2.303828,4.318272,5.183582,4.703271,6.515834,...,2.634693,1.662149,0.023167,1.027306,0.918777,0.408037,0.088542,0.158303,0.246786,3.576926e-02
3,0.030280,16.601984,1.190282,3.603131,1.985821,4.328725,7.982704,9.500140,8.835680,11.644745,...,6.429097,3.430283,0.056623,2.207598,1.593786,0.662720,0.222500,0.423078,0.700987,1.076763e+06
4,0.039917,21.875098,1.489258,4.689429,2.645148,5.347244,9.807538,11.638044,10.914152,14.145230,...,8.668621,4.372804,0.076389,2.852594,1.916814,0.779651,0.302977,0.588125,0.994535,8.314574e-02


In [15]:
check_dataset(dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 76216
Count greater than 1:	 67552
Count less than -1:	 0


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf

def correlation_ledoitwolf_global(exp, dict_groups_subgroups_t):
	dict_groups_subgroups_t_corr = {}
	for group_id, dict_groups in dict_groups_subgroups_t.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			print(group_id, subgroup_id, df_subgroup.shape)
			
			""" import numpy as np
			cov = np.cov(df_subgroup.values, rowvar=False)
			cond = np.linalg.cond(cov)
			print("Condition number:", cond, cond > 1e8) # ill-conditioned if > 1e8 (True, instable) """

			scaler = StandardScaler()
			df_subgroup_scaled = scaler.fit_transform(df_subgroup)
			lw = LedoitWolf()
			lw.fit(df_subgroup_scaled)

			# Matriz de covarianza regularizada
			cov = lw.covariance_
			std = np.sqrt(np.diag(cov))
			corr = cov / np.outer(std, std)
			matrix = pd.DataFrame(corr)
		
			dict_aux[subgroup_id] = matrix
			
			matrix.to_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index=True)
		dict_groups_subgroups_t_corr[group_id] = dict_aux
	return dict_groups_subgroups_t_corr

In [ ]:
# Correlation matrix (partial correlation)

# dict_groups_subgroups_t_corr = correlation_global(exp, dict_groups_subgroups_t)
dict_groups_subgroups_t_corr = exp, dict_groups_subgroups_t)
dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

AR 1 (14, 5444)
AR 2 (14, 5444)
CRS 1 (14, 5444)
CRS 2 (14, 5444)
OSA 1 (14, 5444)
OSA 2 (14, 5444)
LPRD 1 (14, 5444)
LPRD 2 (14, 5444)
SGB 1 (14, 5444)
SGB 2 (14, 5444)
LSNB 1 (14, 5444)
LSNB 2 (14, 5444)
RCC 1 (14, 5444)
RCC 2 (14, 5444)
BC 1 (14, 5444)
BC 2 (14, 5444)
BPH 1 (14, 5444)
BPH 2 (14, 5444)
PCa 1 (14, 5444)
PCa 2 (14, 5444)
PD 1 (14, 5444)
PD 2 (14, 5444)


,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,1.000000,0.382173,0.066751,0.432721,0.205697,0.173611,0.432547,0.432345,0.432405,-0.027422,...,0.125416,-0.021136,0.176524,-0.017027,0.432668,-0.040082,0.373306,0.432048,0.405906,-0.043522
1,0.382173,1.000000,0.259127,0.383676,0.360139,0.338495,0.387631,0.390108,0.389542,-0.013739,...,0.304971,0.044935,0.341162,0.072364,0.384305,-0.067033,0.432270,0.392880,0.428542,-0.092573
2,0.066751,0.259127,1.000000,0.069968,0.407523,0.418085,0.078800,0.084392,0.083059,0.027286,...,0.427014,0.131039,0.416097,0.175116,0.071691,-0.067630,0.272721,0.090286,0.209085,-0.121315
3,0.432721,0.383676,0.069968,1.000000,0.208543,0.176595,0.432629,0.432471,0.432520,-0.027034,...,0.128487,-0.020105,0.179452,-0.015775,0.432698,-0.040477,0.374915,0.432213,0.406990,-0.044408
4,0.205697,0.360139,0.407523,0.208543,1.000000,0.430165,0.216239,0.221134,0.219989,0.012852,...,0.423412,0.110318,0.430940,0.152349,0.209886,-0.078206,0.369530,0.226514,0.323945,-0.127753


In [18]:
# Check correlation matrices

dict_groups_subgroups_t_corr

{'AR': {'1':           0         1         2         3         4         5         6     \
  0     1.000000  0.382173  0.066751  0.432721  0.205697  0.173611  0.432547   
  1     0.382173  1.000000  0.259127  0.383676  0.360139  0.338495  0.387631   
  2     0.066751  0.259127  1.000000  0.069968  0.407523  0.418085  0.078800   
  3     0.432721  0.383676  0.069968  1.000000  0.208543  0.176595  0.432629   
  4     0.205697  0.360139  0.407523  0.208543  1.000000  0.430165  0.216239   
  ...        ...       ...       ...       ...       ...       ...       ...   
  5439 -0.040082 -0.067033 -0.067630 -0.040477 -0.078206 -0.068315 -0.041095   
  5440  0.373306  0.432270  0.272721  0.374915  0.369530  0.348624  0.379118   
  5441  0.432048  0.392880  0.090286  0.432213  0.226514  0.195170  0.432507   
  5442  0.405906  0.428542  0.209085  0.406990  0.323945  0.298115  0.409736   
  5443 -0.043522 -0.092573 -0.121315 -0.044408 -0.127753 -0.123218 -0.046797   
  
            7         8   

In [19]:
check_dataset(dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][1]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 15921570
Count zero:	 0
Count positive:	 13715566
Count greater than 1:	 927
Count less than -1:	 0


In [27]:
def build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=0.3):
    dict_groups_subgroups_t_corr_g = {}
    for group_id, dict_groups in dict_groups_subgroups_t_corr.items():
        dict_aux = {}
        for subgroup_id, df_subgroup in dict_groups.items():
            # Percentil Correlaciones conservadas Densidad esperadas
            # 90	10 %	Alta
			# 95	5 %		Media
			# 97.5	2.5 %	Baja
			# 99	1 %		Muy baja
            threshold = np.percentile(np.abs(df_subgroup), 90)
            
            df_weighted_edges = (df_subgroup.where(np.triu(np.ones(df_subgroup.shape), k=1).astype(bool)).stack())
            df_weighted_edges = df_weighted_edges.dropna().to_frame()
            df_weighted_edges.reset_index(inplace=True)
            df_weighted_edges.columns = ["source", "target", "weight"]
            df_weighted_edges = df_weighted_edges[df_weighted_edges["weight"].abs() >= threshold]
            df_weighted_edges["subgroup"] = [subgroup_id] * len(df_weighted_edges)
            dict_aux[subgroup_id] = df_weighted_edges
            
            df_weighted_edges.to_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
            # G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight"])
            # print(groups_id[i], subgroups_id[groups_id[i]][j], G.number_of_nodes(), G.number_of_edges())
            # nx.write_gexf(G, "experiments/output/{}/preprocessing/graphs/graphs_{}_{}.gexf".format(exp, groups_id[i], subgroups_id[groups_id[i]][j]))
        dict_groups_subgroups_t_corr_g[group_id] = dict_aux
    return dict_groups_subgroups_t_corr_g

In [28]:
# Build graph (corpus graphs)

# dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,source,target,weight,subgroup
0,0,1,0.382173,1
2,0,3,0.432721,1
5,0,6,0.432547,1
6,0,7,0.432345,1
7,0,8,0.432405,1


In [29]:
def create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups, df_join_raw_metadata):	
	for group_id in tqdm(groups_id):
		for subgroup_id in tqdm(subgroups_id[group_id]):
			df_weighted_edges = pd.read_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id))
			# print(df_weighted_edges)
			G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight", "subgroup"])
			dict_id_idx = dict(zip(list(G.nodes()), range(G.number_of_nodes())))
			G = nx.relabel_nodes(G, dict_id_idx)

			df_nodes = dict_df_groups_subgroups[group_id][subgroup_id].loc[list(dict_id_idx.keys())] # A_1.1, A_1.2, A_1.3
			# from IPython.display import display
			# display(df_nodes)

			# nodes, with node features
			metadata = df_join_raw_metadata.loc[df_nodes.index] # Average Rt, Average Mz
			# intensity = df_join_raw_log.loc[df_nodes.index] # A_1.1, A_1.2, A_1.3, A_2.1, ...

			""" e = 1e-8
			mz = metadata.iloc[:, 1].values
			rt = metadata.iloc[:, 0].values
			intensity_mean = df_nodes.mean(axis=1).values
			intensity_std = df_nodes.std(axis=1).values
			intensity_cv = intensity_std / intensity_mean
			presence_ratio = (df_nodes > 0).mean(axis=1)

			mz_log = np.log10(mz + e)
			# intensity_mean_log = np.log10(intensity_mean + e)
			
			# mz_z = (mz_log - mz_log.mean()) / mz_log.std() # z-score
			rt_z = (rt - rt.mean()) / rt.std() # z-score
			# intensity_mean_z = (intensity_mean_log - intensity_mean_log.mean()) / intensity_mean_log.std() # z-score

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": mz_log,
				"rt": rt_z,
				"intensity_mean": intensity_mean,
				"intensity_std": intensity_std,
				"intensity_cv": intensity_cv,
				"presence_ratio": presence_ratio
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			# df_node_features.insert(0, "idx", list(dict_id_idx.values()))
			# df_node_features.insert(1, "id", list(dict_id_idx.keys()))
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features) """

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": metadata.iloc[:, 1].values,
				"rt": metadata.iloc[:, 0].values,
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features)

			# edges
			edges = list(G.edges())
			df_edges = pd.DataFrame(edges, columns=["source", "target"])
			df_edges["weight"] = [G.get_edge_data(*edge)["weight"] for edge in edges]
			df_edges["subgroup"] = [G.get_edge_data(*edge)["subgroup"] for edge in edges]
			df_edges.to_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)


In [30]:
# create dataset - nodes/edge data for PyTorch Geometric/DGL framework

# IMPORTANT
dict_df_groups_subgroups_ = split_groups_subgroups(df_join_raw_intensity, groups_id, subgroups_id) # Important (intesities without Log)

for data_variation in data_variations:
	if data_variation == "none":
		create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, df_join_raw_metadata)
	else:
		# dynamic graph to static graph
		create_graph_data_directed_variation(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, data_variation)

100%|██████████| 11/11 [02:37<00:00, 14.34s/it]


In [31]:
# details
list_details = []
	
for group_id in groups_id:
	subgroups_id_ = []
	for data_variation in data_variations:
		if data_variation == "none":
			subgroups_id_ += subgroups_id[group_id]
		else:
			subgroups_id_ += [data_variation]
	# print(subgroups)
	
	for subgroup_id_ in subgroups_id_:
		try:
			df_edges = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id_))

			G = nx.from_pandas_edgelist(df_edges.iloc[:, [0, 1]])
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), nx.density(G), np.nan, nx.is_connected(G)])
		except:
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), np.nan, np.nan, np.nan])

df_details = pd.DataFrame(list_details, columns=["Group", "Subgroup", "Num. nodes", "Num. edges", "Density", "Diameter", "Is connected"])
df_details.to_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp), index=False)

df_details = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp))
df_details

,Group,Subgroup,Num. nodes,Num. edges,Density,Diameter,Is connected
0,AR,1,5444,1479135,0.099835,NaN,True
1,AR,2,5444,1479135,0.099835,NaN,True
2,CRS,1,5444,1479135,0.099835,NaN,True
3,CRS,2,5444,1479135,0.099835,NaN,True
4,OSA,1,5444,1479135,0.099835,NaN,True
5,OSA,2,5444,1479135,0.099835,NaN,True
6,LPRD,1,5444,1479135,0.099835,NaN,True
7,LPRD,2,5444,1479135,0.099835,NaN,True
8,SGB,1,5444,1479135,0.099835,NaN,True
9,SGB,2,5444,1479135,0.099835,NaN,True
